# InfiniGen — mlp-eviction smoke test (Colab GPU)

This notebook prepares a Colab GPU runtime, installs dependencies, clones the repository branch `notebook-stable`, and runs a fast smoke test of the `mlp` eviction policy using `accuracy/perplexity/run_single_mlp.py`.

Checklist:
- GPU runtime recommended (Runtime > Change runtime type > GPU).
- Installs PyTorch + Transformers (attempts CUDA wheel).
- Clones repo into `/content/InfiniGen`.
- Runs fast smoke test: OPT 6.7b, wikitext2, fast mode (short seq, 1 sample).
- Logs saved under `accuracy/perplexity/mlp_single_logs/`.

In [ ]:
%%bash
set -e
echo '--- Colab GPU check ---'
if command -v nvidia-smi >/dev/null 2>&1; then
  nvidia-smi || true
else
  echo 'nvidia-smi not found; the runtime may not have a GPU or drivers available.';
fi
python -c "import torch,sys; print('torch', getattr(torch,'__version__',None), 'cuda:', torch.cuda.is_available())" 2>/dev/null || true

In [ ]:
%%bash
set -e
echo '--- Upgrade pip and install transformers/sentencepiece ---'
python -m pip install -q --upgrade pip
python -m pip install -q transformers==4.35.0 sentencepiece || python -m pip install -q transformers sentencepiece
echo 'Attempt to install CUDA-enabled PyTorch (cu118) then fallback to CPU'
python -c "import torch,sys; print('torch', getattr(torch,'__version__',None))" 2>/dev/null || true
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 || pip install -q torch torchvision torchaudio
python -c "import torch; print('torch', torch.__version__, 'cuda_available:', torch.cuda.is_available())"

In [ ]:
%%bash
set -e
REPO_URL=https://github.com/pinhas019/InfiniGen.git
BRANCH=pinhas-mlp-agents
WORKDIR=/content/InfiniGen
if [ -d "$WORKDIR" ]; then
  echo "Existing checkout found at $WORKDIR, skipping clone (remove to re-clone)";
else
  echo "Cloning ${REPO_URL} (branch ${BRANCH})...";
  git clone --branch ${BRANCH} ${REPO_URL} ${WORKDIR};
fi
cd $WORKDIR
echo 'Installing repo requirements (if present)...'
if [ -f requirements.txt ]; then
  python -m pip install -q -r requirements.txt || true
else
  echo 'No requirements.txt at repo root; continuing.'
fi
echo 'Listing accuracy/perplexity contents:'
ls -la accuracy/perplexity || true


## Optional: Mount Google Drive for LLaMA weights
If you plan to run LLaMA experiments, mount Drive and set `LLAMA_PATH` to the directory that contains `llama-2-7b` and `llama-2-13b` directories.
Example:
```python
from google.colab import drive
drive.mount('/content/drive')
LLAMA_PATH = '/content/drive/MyDrive/path/to/llama_models'
```

In [ ]:
%%bash
set -e
cd /content/InfiniGen/accuracy/perplexity
echo 'Running the single-model mlp eviction smoke test (fast mode): OPT 6.7b on wikitext2'
python run_single_mlp.py --model-type opt --size 6.7b --dataset wikitext2 --fast || true
echo 'Logs written to ./mlp_single_logs/'
ls -la mlp_single_logs || true

In [ ]:
%%bash
set -e
LOG_DIR=/content/InfiniGen/accuracy/perplexity/mlp_single_logs
echo 'Tail of last log files (if any):'
for f in $(ls -1t ${LOG_DIR} 2>/dev/null | head -n 5); do
  echo '---' ${f} '---';
  tail -n 300 "${LOG_DIR}/${f}" || true;
done
echo 'If no logs are present, inspect previous cells for errors (likely missing model files or dependencies).'